# Assignment 4
In this assignment you will be using the dataset released by The Department of Transportation. This dataset lists flights that occurred in 2015, along with other information such as delays, flight time etc.

In this assignment, you will be showing good practices to manipulate data using Python's most popular libraries to accomplish the following:

- cleaning data with pandas
- make specific changes with numpy
- handling date-related values with datetime

Note: please consider the flights departing from BOS, JFK, SFO and LAX.

Each question is equally weighted for the total grade.

In [2]:
import os
import pandas as pd
import pandas.api.types as ptypes
import numpy as np
import datetime as dt
from mads.lib.path import assets

# Retrieve data
file_flights = assets.find("flights.csv")
file_airports = assets.find("airports.csv")
file_airlines = assets.find("airlines.csv")
flights_df_raw = pd.read_csv(file_flights, low_memory=False)
airports_df = pd.read_csv(file_airports)
airlines_df = pd.read_csv(file_airlines)

## Question 1: Data Preprocessing
For this question, perform the following:

- remove rows with missing values
- keep flights departing from airports (ORIGIN_AIRPORT) that we want to look at (BOS, JFK, SFO and LAX)
- filter out the flights that have more than 1 day delay (DEPARTURE_DELAY)
- convert FLIGHT_NUMBER column type to string
- SCHEDULED_DEPARTURE is coded as a float where the first two digits indicate the hour and the last two indicate the minutes. Convert this column to datetime format by combining existing columns DAY, MONTH, YEAR and SCHEDULED_DEPARTURE
- add IS_DELAYED column by considering any flight with at least 15 minutes delay (DEPARTURE_DELAY) are delayed, and any other flight is not delayed
- remove YEAR, MONTH, DAY columns

In [3]:
flights_df_raw[flights_df_raw["DEPARTURE_DELAY"] > 1440].head()

,YEAR,MONTH,DAY,ORIGIN_AIRPORT,DESTINATION_AIRPORT,AIRLINE,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_DELAY
145439,2015,1,18,LAS,LAX,AA,224,1130,1604.0
1225250,2015,5,21,LAX,STL,AA,1258,610,1492.0
1518916,2015,6,22,SFO,DFW,AA,1454,515,1496.0
2567756,2015,11,15,MCO,JFK,AA,290,1215,1536.0
2667908,2015,11,27,DTW,ORD,AA,2559,1027,1631.0


In [102]:
def data_preprocess(flights_df):
    # YOUR CODE HERE
    df = flights_df.dropna()
    df = df[df['ORIGIN_AIRPORT'].isin(['BOS', 'JFK', 'SFO', 'LAX'])]
    df = df[df['DEPARTURE_DELAY'] < 1440]
    df['FLIGHT_NUMBER'] = df['FLIGHT_NUMBER'].astype(str)
    #scheduled_departure
    df['SCHEDULED_DEPARTURE'] = pd.to_datetime(
    df[['YEAR', 'MONTH', 'DAY']].astype(str).agg('-'.join, axis=1) + ' ' +
    df['SCHEDULED_DEPARTURE'].astype(str).str.zfill(4),
    format='%Y-%m-%d %H%M')
    #
    df['IS_DELAYED'] = df['DEPARTURE_DELAY'].apply(lambda x: 1 if x >= 15 else 0)
    df = df.drop(columns=['YEAR', 'MONTH', 'DAY'])
    return df

In [103]:
#flights_df = data_preprocess(flights_df_raw.copy())
#flights_df.head()

In [104]:
flights_df = data_preprocess(flights_df_raw.copy())
assert len(flights_df) == 535744, "Q1: There should be 535744 observations in the flights dataframe"


In [ ]:
flights_df['FLIGHT_N

## Question 2
Merge flights_df dataframe with airports_df dataframe and return the number of departing flights (*NUM_FLIGHTS*) per airport (*IATA_CODE*) across the year.

In [76]:
#flights_df.head()

In [77]:
#airports_df.head()

In [78]:
#flights_df['ORIGIN_AIRPORT'].unique() 

In [79]:
def flights_per_airport(flights_df, airports_df):
    flights_df = data_preprocess(flights_df.copy())
    merged_df = pd.merge(flights_df, airports_df, how='left', left_on='ORIGIN_AIRPORT', right_on='IATA_CODE')
    #print(merged_df.shape)
    #print(merged_df['ORIGIN_AIRPORT'].unique()) 
    df = merged_df.groupby('IATA_CODE').size().reset_index(name='NUM_FLIGHTS').set_index('IATA_CODE')

    #print(df.shape)
    return df

In [80]:
#num_flights_df=flights_per_airport(flights_df_raw.copy(), airports_df.copy())

In [81]:
#num_flights_df.head()

In [82]:
num_flights_df=flights_per_airport(flights_df_raw.copy(), airports_df.copy())

assert num_flights_df.shape==(4,1), "Shape of DataFrame should be (4,1)"
assert num_flights_df.columns[0]=='NUM_FLIGHTS', "DataFrame should have a column which is called NUM_FLIGHTS"
assert num_flights_df.loc["BOS", "NUM_FLIGHTS"] == 105276, "The NUM_FLIGHTS for BOS is wrong"


## Question 3
For this question, find the top three airline names which have high number of flights and the least percentage of delay compared to other airlines. The result should be a dataframe which has three columns *AIRLINE_NAME*, *NUM_FLIGHTS* and *PERC_DELAY*.

Hint:
- percentage of delay for each airline is obtained using groupby and apply methods
- merge flights_df with airlines_df to get the names of top three airlines

In [83]:
#num_flights_df.head()

In [84]:
#flights_df_raw.head()

In [85]:
#airlines_df.head()

In [86]:
#airlines_df_copy = airlines_df.rename(columns={'AIRLINE': 'AIRLINE_NAME'})
#airlines_df_copy.head()

In [87]:
#merged_df = pd.merge(data_preprocess(flights_df_raw.copy()), airlines_df_copy.copy(), how='left', left_on='AIRLINE' , right_on='IATA_CODE')
#df = merged_df.groupby('AIRLINE').size().reset_index(name='NUM_FLIGHTS').set_index('AIRLINE')

#df = (merged_df.groupby('AIRLINE')
#      .agg(NUM_FLIGHTS=('AIRLINE', 'count')
#           ,PERC_DELAY=('IS_DELAYED', 'mean')
#           ,AIRLINE_NAME=('AIRLINE_NAME', 'first'))
#      .rename_axis('AIRLINE'))
#
#df = df.sort_values(by=['NUM_FLIGHTS', 'PERC_DELAY', 'AIRLINE_NAME'], ascending=[False, False, True])
#
#df = df.head(3)

In [88]:
def top_three_airlines(flights_df, airlines_df):
    # YOUR CODE HERE
    airlines_df_copy = airlines_df.rename(columns={'AIRLINE': 'AIRLINE_NAME'})

    merged_df = pd.merge(data_preprocess(flights_df), airlines_df_copy, how='left', left_on='AIRLINE' , right_on='IATA_CODE')

    df = (merged_df.groupby('AIRLINE')
          .agg(NUM_FLIGHTS=('AIRLINE', 'count')
               ,PERC_DELAY=('IS_DELAYED', 'mean')
               ,AIRLINE_NAME=('AIRLINE_NAME', 'first'))
          .reset_index()
          .drop(columns='AIRLINE'))

    df = df.sort_values(by=['NUM_FLIGHTS', 'PERC_DELAY', 'AIRLINE_NAME'], ascending=[False, False, True])

    df = df.head(3).reset_index(drop=True)
    return df

In [89]:
#top_three_airlines_df = top_three_airlines(flights_df_raw.copy(), airlines_df.copy())
#top_three_airlines_df.head()
#print(top_three_airlines_df.loc[0, 'AIRLINE_NAME'])

In [90]:
top_three_airlines_df = top_three_airlines(flights_df_raw.copy(), airlines_df.copy())

assert sorted(list(top_three_airlines_df.columns)) == sorted(['NUM_FLIGHTS', 'PERC_DELAY', 'AIRLINE_NAME']), "Dataframe doesn't have required columns"
assert top_three_airlines_df.loc[0, 'AIRLINE_NAME'] == 'United Air Lines Inc.', "Top airline name doesn't match"


## Question 4
For this question, obtain the monthly percentage of delays for each *ORIGIN_AIRPORT*.

Example Result:

         MONTH     BOS     JFK     LAX     SFO
    0   January  0.1902  0.2257  0.1738  0.xxxx
    1  February  0.3248  0.xxxx  0.xxxx  0.xxxx
    2     March  0.1984  0.xxxx  0.xxxx  0.xxxx
    3     April  0.xxxx  0.xxxx  0.xxxx  0.xxxx

In [96]:
#df = data_preprocess(flights_df_raw.copy())
#df.head()

In [97]:
#flights_df['MONTH'].unique()

In [98]:
#df.shape

In [99]:
def monthly_airport_delays(flights_df):
    # YOUR CODE HERE
    flights_df = data_preprocess(flights_df)
    flights_df['MONTH'] = flights_df['SCHEDULED_DEPARTURE'].dt.strftime('%B')

    month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
    flights_df['MONTH'] = pd.Categorical(flights_df['MONTH'], categories=month_order, ordered=True)

    df = flights_df.pivot_table(index='MONTH', columns='ORIGIN_AIRPORT', values='IS_DELAYED', aggfunc='mean')
    #print(df)
    df.columns.name = None
    df = df.reset_index()
    df['MONTH'] = df['MONTH'].astype(str)
    df = df.round(4)
    return df

In [100]:
monthly_airport_delays_df = monthly_airport_delays(flights_df_raw.copy())


In [101]:
#monthly_airport_delays_df